In [ ]:
dbutils.widgets.text("catalog", "adb_adventureworks_project")
dbutils.widgets.text("silver_schema", "silver_dev")
dbutils.widgets.text("gold_schema", "gold_dev")

## Check for VERSION CONFLICTS

In [ ]:
from pyspark.sql import functions as F

# Read environment parameters
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")

silver_table = f"{catalog}.{silver_schema}.sales_silver_table"

# Read Silver data
df = spark.table(silver_table)

# Detect duplicate business key + version
version_conflicts = (
    df.groupBy(
      "OrderNumber",
      "OrderLineItem",
      F.coalesce(
        F.col("RecordVersion"),
        F.lit(1)
      ).alias("EffectiveRecordVersion")
    )
    .count()
    .filter(F.col("count") > 1)
)

# Fail fast if conflicts exist
conflict_samples = version_conflicts.limit(5).collect()

if conflict_samples:
    raise ValueError(
        "Sales version conflicts detected! "
        f"Examples: {[row.asDict() for row in conflict_samples]}"
    )

print("PASS: No sales version conflicts detected.")

## Fact_Sales

In [0]:
%sql

MERGE INTO IDENTIFIER(:catalog || '.' || :gold_schema || '.fact_sales') AS target
USING (
    SELECT
        s.OrderNumber,
        s.OrderLineItem,
        s.OrderDate,
        s.StockDate,
        s.ProductKey,
        s.CustomerKey,
        s.TerritoryKey,
        s.OrderQuantity,

        CAST(p.ProductPrice AS DECIMAL(18,4)) AS UnitPrice,
        CAST(p.ProductCost AS DECIMAL(18,4)) AS UnitCost,

        CAST(s.OrderQuantity * p.ProductPrice AS DECIMAL(18,4)) AS EstimatedRevenue,
        CAST(s.OrderQuantity * p.ProductCost AS DECIMAL(18,4)) AS EstimatedTotalCost,

        CAST(s.OrderQuantity * (p.ProductPrice - p.ProductCost)AS DECIMAL(18,4)) AS EstimatedGrossProfit

    FROM 
        -- Read the current version of each order-line instead of the entire history directly in Silver
        IDENTIFIER(:catalog || '.' || :silver_schema || '.sales_current_view') s    
    INNER JOIN 
        IDENTIFIER(:catalog || '.' || :silver_schema || '.products_silver_table') p
    ON 
        s.ProductKey = p.ProductKey
) AS source

ON target.OrderNumber = source.OrderNumber
AND target.OrderLineItem = source.OrderLineItem

WHEN MATCHED AND (
    NOT (target.OrderDate <=> source.OrderDate)                 -- null-safe equality operator
    OR NOT (target.StockDate <=> source.StockDate)
    OR NOT (target.ProductKey <=> source.ProductKey)
    OR NOT (target.CustomerKey <=> source.CustomerKey)
    OR NOT (target.TerritoryKey <=> source.TerritoryKey)
    OR NOT (target.OrderQuantity <=> source.OrderQuantity)
    OR NOT (target.UnitPrice <=> source.UnitPrice)
    OR NOT (target.UnitCost <=> source.UnitCost)
) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


In [0]:
%sql

SELECT
    'fact_sales' AS TableName,
    ROUND(SUM(EstimatedRevenue), 2) AS TotalRevenue
FROM adb_adventureworks_project.gold.fact_sales

UNION ALL

SELECT
    'agg_monthly_sales' AS TableName,
    ROUND(SUM(EstimatedRevenue), 2) AS TotalRevenue
FROM adb_adventureworks_project.gold.agg_monthly_sales;

